In [16]:
from ioMicro import *
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [17]:
data_folder = r'Z:\Zane_20CRE\9_14_2024__T7_20CRE'

In [18]:
# map all the hybes
hybes =  glob.glob(data_folder+os.sep+'H*') + glob.glob(data_folder+os.sep+'D*')
# map all the fovs
fovs = [os.path.basename(fl)for fl in glob.glob(hybes[0]+os.sep+'*.zarr')]
def get_Hi(fld): 
    try: return int(os.path.basename(fld)[1:]) 
    except: return -1
    
hybes = np.array(hybes)[np.argsort([get_Hi(hybe) for hybe in hybes])]

In [21]:
hybes[0]

'Z:\\Zane_20CRE\\9_14_2024__T7_20CRE\\HGFP'

In [22]:
def drift_correction(hybes,fov,save_file,iiref=None,redo=False,ssz=20):
    if iiref is None: iiref = len(hybes)//2
    dic_drift = {}
    if os.path.exists(save_file):
        dic_drift = pickle.load(open(save_file,'rb'))
    tags = [os.path.basename(hybe) for hybe in hybes]
    redo = ~np.all([tag in dic_drift for tag in tags])
    if redo:
        #Load the reference dapi image
        fl1 = hybes[iiref]+os.sep+fov
        im1 = read_im(fl1)
        ncols,sz,sx,sy = im1.shape
        im1dapi = np.array(im1[-1][(sz-ssz)//2:(sz+ssz)//2],dtype=np.float32)
        ###Perform drift

        for hybe in hybes:
            fl2 = hybe+os.sep+fov
            tag = os.path.basename(hybe)
            if not (tag in dic_drift):
                im2 = read_im(fl2)
                ncols,sz,sx,sy = im1.shape
                

                im2dapi = np.array(im2[-1][(sz-ssz)//2:(sz+ssz)//2],dtype=np.float32)
                txyz,txyzs = get_txyz(im1dapi,im2dapi,sz_norm=30,sz = 500,nelems=5)
                dic_drift[tag]=[txyz,txyzs]
                #print(txyz,txyzs)
                pickle.dump(dic_drift,open(save_file,'wb'))

In [23]:
for fov in tqdm(fovs):
    analysis_folder = r'Z:\Zane_20CRE\9_14_2024__T7_20CRE\drifts_polyA'
    save_file = analysis_folder+os.sep+fov.split('.')[0]+'_drift.pkl'
    
    drift_correction(hybes,fov,save_file,iiref=0,redo=False,ssz=20)

100%|████████████████████████████████████████████████████████████████████████████| 184/184 [12:17:02<00:00, 240.34s/it]
